In [1]:
from __future__ import annotations

"""
Experiment 3 for the 3D topological Vicsek model
=================================================

This version is explicitly aligned with the user's Experiment 1 and Experiment 2 code:

- Dynamics / noise amplitude:
    u_new = normalize(A + sqrt(2 D dt) * eta)
- Main cost proxy Sdot (same style as Experiment 1):
    sdot = mean_i ||u_new - A||^2 / (2 D dt)
- Main response definition:
    chi_phi = N * ( <phi^2> - <phi>^2 )
- Alternative cost proxy:
    c_mis = mean_i (1 - u_i · A_i)
- Alternative response proxies:
    chi_conn, chi_xi = xi^3, chi_C = 4 pi int_0^xi r^2 C(r) dr
- Statistical extraction:
    bootstrap over seeds for D_c, D_opt, Delta, and valley depth
- Boundary / resolution defense:
    local dense scan and boundary extension

The file is notebook-friendly (# %% cells) and also supports CLI.
"""



"\nExperiment 3 for the 3D topological Vicsek model\n=================================================\n\nThis version is explicitly aligned with the user's Experiment 1 and Experiment 2 code:\n\n- Dynamics / noise amplitude:\n    u_new = normalize(A + sqrt(2 D dt) * eta)\n- Main cost proxy Sdot (same style as Experiment 1):\n    sdot = mean_i ||u_new - A||^2 / (2 D dt)\n- Main response definition:\n    chi_phi = N * ( <phi^2> - <phi>^2 )\n- Alternative cost proxy:\n    c_mis = mean_i (1 - u_i · A_i)\n- Alternative response proxies:\n    chi_conn, chi_xi = xi^3, chi_C = 4 pi int_0^xi r^2 C(r) dr\n- Statistical extraction:\n    bootstrap over seeds for D_c, D_opt, Delta, and valley depth\n- Boundary / resolution defense:\n    local dense scan and boundary extension\n\nThe file is notebook-friendly (# %% cells) and also supports CLI.\n"

In [2]:
import os
import math
import json
import time
import hashlib
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")

import jax
import jax.numpy as jnp
from jax import lax, random as jr




In [3]:
Array = jnp.ndarray
EPS = 1e-12


def report_jax_backend(require_gpu: bool = True) -> List[str]:
    devices = jax.devices()
    desc = [f"{d.platform}:{getattr(d, 'device_kind', type(d).__name__)}" for d in devices]
    print("JAX devices:", desc)
    if require_gpu and not any(d.platform == "gpu" for d in devices):
        raise RuntimeError(
            "No JAX GPU device detected. Please make sure this notebook/kernel uses the GPU-enabled JAX environment."
        )
    return desc


def normalize(v: Array, axis: int = -1, eps: float = EPS) -> Array:
    return v / jnp.maximum(jnp.linalg.norm(v, axis=axis, keepdims=True), eps)


def minimum_image(delta: Array, L: float) -> Array:
    return delta - L * jnp.round(delta / L)


def wrap_positions(x: Array, L: float) -> Array:
    return jnp.mod(x, L)


def stable_boot_seed(*items: Any) -> int:
    token = "|".join(str(x) for x in items).encode("utf-8")
    return int.from_bytes(hashlib.blake2b(token, digest_size=8).digest(), "little") % (2**32)


def system_size(N: int, rho: float) -> float:
    return float((N / rho) ** (1.0 / 3.0))




In [4]:
@dataclass
class Exp3Config:
    # physical parameters (aligned with Exp1/Exp2)
    rho: float = 1.0
    v0: float = 0.05
    dt: float = 1.0
    include_self_in_alignment: bool = False

    # runtime / sampling
    burnin_steps: int = 5000
    sample_stride: int = 50
    n_samples: int = 160

    # seed batching on GPU
    seed_batch_size_1024: int = 8
    seed_batch_size_2048: int = 4

    # correlation estimation
    corr_nbins: int = 32
    corr_sample_size: int = 512
    corr_num_frames: int = 8
    corr_rmax_fraction: float = 0.50

    # bootstrap / valley extraction
    bootstrap_reps: int = 2000
    valley_side_points: int = 2

    # io / safety
    output_dir: str = "results_exp3_gpu_jax"
    resume: bool = True
    require_gpu: bool = True

    @property
    def total_steps(self) -> int:
        return self.burnin_steps + self.sample_stride * self.n_samples




In [5]:
def seed_batch_size_for_N(N: int, cfg: Exp3Config) -> int:
    return cfg.seed_batch_size_1024 if N <= 1024 else cfg.seed_batch_size_2048


def ensure_dirs(output_dir: str) -> Dict[str, str]:
    paths = {
        "root": output_dir,
        "seed": os.path.join(output_dir, "seed_metrics"),
        "agg": os.path.join(output_dir, "aggregates"),
        "boot": os.path.join(output_dir, "bootstrap"),
        "fig": os.path.join(output_dir, "figures"),
        "meta": os.path.join(output_dir, "meta"),
    }
    for p in paths.values():
        os.makedirs(p, exist_ok=True)
    return paths




In [6]:
def pairwise_sq_distances_batched(x: Array, L: float) -> Array:
    delta = x[:, :, None, :] - x[:, None, :, :]
    delta = minimum_image(delta, L)
    return jnp.sum(delta * delta, axis=-1)


def knn_indices_batched(x: Array, k: int, L: float) -> Array:
    B, N, _ = x.shape
    d2 = pairwise_sq_distances_batched(x, L)
    d2 = d2 + 1e10 * jnp.eye(N, dtype=d2.dtype)[None, :, :]
    _, idx = lax.top_k(-d2, k)
    return idx.astype(jnp.int32)


def gather_neighbors_batched(u: Array, idx: Array) -> Array:
    def gather_one(ub: Array, ib: Array) -> Array:
        return ub[ib]
    return jax.vmap(gather_one, in_axes=(0, 0))(u, idx)


def local_alignment_field_batched(u: Array, x: Array, k: int, L: float, include_self: bool) -> Tuple[Array, Array]:
    idx = knn_indices_batched(x, k, L)
    nbr_u = gather_neighbors_batched(u, idx)
    if include_self:
        A = (u + jnp.sum(nbr_u, axis=2)) / float(k + 1)
    else:
        A = jnp.mean(nbr_u, axis=2)
    A = normalize(A)
    return A, idx


def vicsek_step_batched(
    state: Tuple[Array, Array],
    key: Array,
    D: float,
    k: int,
    L: float,
    v0: float,
    dt: float,
    include_self: bool,
) -> Tuple[Tuple[Array, Array], Tuple[Array, Array, Array, Array, Array]]:
    """
    Returns:
      new_state,
      (phi, mvec, sdot_step, cmis_step, idx)
    """
    x, u = state
    A, idx = local_alignment_field_batched(u, x, k, L, include_self)
    eta = jr.normal(key, shape=u.shape, dtype=u.dtype)
    u_new = normalize(A + jnp.sqrt(2.0 * D * dt) * eta)
    x_new = wrap_positions(x + v0 * dt * u_new, L)

    mvec = jnp.mean(u_new, axis=1)
    phi = jnp.linalg.norm(mvec, axis=-1)

    # Main cost proxy aligned with Experiment 1
    sdot_step = jnp.mean(jnp.sum((u_new - A) ** 2, axis=-1), axis=1) / jnp.maximum(2.0 * D * dt, EPS)

    # Alternative cost proxy requested for Experiment 3
    cmis_step = jnp.mean(1.0 - jnp.sum(u * A, axis=-1), axis=1)

    return (x_new, u_new), (phi, mvec, sdot_step, cmis_step, idx)




In [7]:
_RUNNER_CACHE: Dict[Tuple, Any] = {}


def cfg_signature(cfg: Exp3Config) -> Tuple:
    return (
        cfg.rho,
        cfg.v0,
        cfg.dt,
        cfg.include_self_in_alignment,
        cfg.burnin_steps,
        cfg.sample_stride,
        cfg.n_samples,
        cfg.corr_nbins,
        cfg.corr_sample_size,
        cfg.corr_num_frames,
        cfg.corr_rmax_fraction,
    )


def make_condition_runner(N: int, k: int, batch_size: int, cfg: Exp3Config):
    key = (N, k, batch_size) + cfg_signature(cfg)
    if key in _RUNNER_CACHE:
        return _RUNNER_CACHE[key]

    L = float(system_size(N, cfg.rho))

    @jax.jit
    def run_condition(base_key: Array, D: float):
        key_x, key_u, key_run = jr.split(base_key, 3)
        x0 = jr.uniform(key_x, shape=(batch_size, N, 3), minval=0.0, maxval=L, dtype=jnp.float32)
        u0 = normalize(jr.normal(key_u, shape=(batch_size, N, 3), dtype=jnp.float32))
        state0 = (x0, u0)

        burn_keys = jr.split(key_run, cfg.burnin_steps)

        def burn_body(state, step_key):
            state, _ = vicsek_step_batched(
                state, step_key, D, k, L, cfg.v0, cfg.dt, cfg.include_self_in_alignment
            )
            return state, None

        state_burn, _ = lax.scan(burn_body, state0, burn_keys)

        sample_keys = jr.split(jr.fold_in(key_run, 999), cfg.n_samples)

        def one_sample(state, outer_key):
            stride_keys = jr.split(outer_key, cfg.sample_stride)

            def stride_body(carry, step_key):
                st, sdot_acc, cmis_acc, idx_last = carry
                st, aux = vicsek_step_batched(
                    st, step_key, D, k, L, cfg.v0, cfg.dt, cfg.include_self_in_alignment
                )
                phi, mvec, sdot, cmis, idx = aux
                return (st, sdot_acc + sdot, cmis_acc + cmis, idx), (phi, mvec)

            zero = jnp.zeros((batch_size,), dtype=jnp.float32)
            dummy_idx = jnp.zeros((batch_size, N, k), dtype=jnp.int32)
            (state_out, sdot_acc, cmis_acc, idx_last), (phi_seq, mvec_seq) = lax.scan(
                stride_body, (state, zero, zero, dummy_idx), stride_keys
            )
            x_out, u_out = state_out
            return state_out, (
                x_out,
                u_out,
                phi_seq[-1],
                mvec_seq[-1],
                sdot_acc / float(cfg.sample_stride),
                cmis_acc / float(cfg.sample_stride),
                idx_last,
            )

        _, samples = lax.scan(one_sample, state_burn, sample_keys)
        x_s, u_s, phi_s, mvec_s, sdot_s, cmis_s, idx_s = samples

        return {
            "x_samples": jnp.swapaxes(x_s, 0, 1),
            "u_samples": jnp.swapaxes(u_s, 0, 1),
            "phi_samples": jnp.swapaxes(phi_s, 0, 1),
            "mvec_samples": jnp.swapaxes(mvec_s, 0, 1),
            "sdot_samples": jnp.swapaxes(sdot_s, 0, 1),
            "cmis_samples": jnp.swapaxes(cmis_s, 0, 1),
            "nbr_idx_samples": jnp.swapaxes(idx_s, 0, 1),
            "L": jnp.asarray(L, dtype=jnp.float32),
        }

    _RUNNER_CACHE[key] = run_condition
    return run_condition




In [8]:
def first_zero_crossing_linear(r: np.ndarray, c: np.ndarray) -> float:
    valid = np.isfinite(r) & np.isfinite(c)
    r = np.asarray(r[valid], dtype=float)
    c = np.asarray(c[valid], dtype=float)
    if r.size < 2:
        return float("nan")
    idx = np.where(c <= 0.0)[0]
    if len(idx) == 0:
        return float(r[-1])
    i = int(idx[0])
    if i == 0:
        return float(r[0])
    x0, x1 = r[i - 1], r[i]
    y0, y1 = c[i - 1], c[i]
    if abs(y1 - y0) < 1e-12:
        return float(x1)
    return float(x0 + (0.0 - y0) * (x1 - x0) / (y1 - y0))


def trapz_compat(y: np.ndarray, x: np.ndarray) -> float:
    if hasattr(np, "trapezoid"):
        return float(np.trapezoid(y, x=x))
    return float(np.trapz(y, x=x))


def integrate_positive_until_xi(r: np.ndarray, c: np.ndarray, xi: float) -> float:
    r = np.asarray(r, dtype=float)
    c = np.asarray(c, dtype=float)
    mask = np.isfinite(r) & np.isfinite(c) & (r <= xi)
    if mask.sum() < 2:
        return 0.0
    r_use = r[mask]
    c_use = c[mask]
    return trapz_compat(4.0 * math.pi * (r_use ** 2) * c_use, r_use)


def radial_connected_correlation_subsample(
    x: np.ndarray,
    u: np.ndarray,
    L: float,
    nbins: int,
    rmax_fraction: float,
    sample_size: int,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, np.ndarray]:
    N = x.shape[0]
    M = min(N, sample_size)
    idx = rng.choice(N, size=M, replace=False)
    xs = x[idx]
    us = u[idx]
    du = us - us.mean(axis=0, keepdims=True)
    denom = max(float(np.mean(np.sum(du * du, axis=1))), 1e-12)

    dx = xs[:, None, :] - xs[None, :, :]
    dx = dx - L * np.round(dx / L)
    dist = np.linalg.norm(dx, axis=-1)
    dot = (du @ du.T) / denom

    iu, ju = np.triu_indices(M, k=1)
    d = dist[iu, ju]
    c = dot[iu, ju]

    rmax = float(rmax_fraction) * float(L)
    edges = np.linspace(0.0, rmax, nbins + 1, dtype=np.float64)
    centers = 0.5 * (edges[:-1] + edges[1:])
    bin_idx = np.searchsorted(edges, d, side="right") - 1
    valid = (bin_idx >= 0) & (bin_idx < nbins) & np.isfinite(c)

    sums = np.zeros(nbins, dtype=np.float64)
    hits = np.zeros(nbins, dtype=np.float64)
    np.add.at(sums, bin_idx[valid], c[valid])
    np.add.at(hits, bin_idx[valid], 1.0)
    Cr = np.divide(sums, np.maximum(hits, 1.0))
    Cr[hits <= 0] = np.nan
    return centers.astype(np.float32), Cr.astype(np.float32)


def compute_seed_correlation_metrics(
    x_samples: np.ndarray,
    u_samples: np.ndarray,
    L: float,
    cfg: Exp3Config,
    rng_seed: int,
) -> Dict[str, Any]:
    S = x_samples.shape[0]
    pick = np.linspace(0, S - 1, min(S, cfg.corr_num_frames), dtype=int)
    rng = np.random.default_rng(rng_seed)

    r_list: List[np.ndarray] = []
    c_list: List[np.ndarray] = []
    for t in pick:
        r_t, c_t = radial_connected_correlation_subsample(
            x_samples[t],
            u_samples[t],
            L=L,
            nbins=cfg.corr_nbins,
            rmax_fraction=cfg.corr_rmax_fraction,
            sample_size=cfg.corr_sample_size,
            rng=rng,
        )
        r_list.append(r_t)
        c_list.append(c_t)

    r = np.nanmean(np.stack(r_list, axis=0), axis=0)
    C_r = np.nanmean(np.stack(c_list, axis=0), axis=0)
    xi = first_zero_crossing_linear(r, C_r)
    chi_xi = float(xi ** 3) if np.isfinite(xi) else float("nan")
    chi_C = integrate_positive_until_xi(r, C_r, xi) if np.isfinite(xi) else float("nan")

    return {
        "r": r,
        "C_r": C_r,
        "xi": float(xi),
        "chi_xi": float(chi_xi),
        "chi_C": float(chi_C),
    }




In [9]:
def compute_basic_seed_metrics(sim_out: Dict[str, np.ndarray], N: int) -> Dict[str, np.ndarray]:
    phi = sim_out["phi_samples"]
    mvec = sim_out["mvec_samples"]
    sdot = sim_out["sdot_samples"]
    cmis = sim_out["cmis_samples"]

    # Main response definition from Exp1 / Exp3 text
    phi_mean = np.mean(phi, axis=1)
    chi_phi = N * (np.mean(phi ** 2, axis=1) - phi_mean ** 2)

    # Vector connected fluctuation susceptibility proxy
    m_center = mvec - np.mean(mvec, axis=1, keepdims=True)
    chi_conn = N * np.mean(np.sum(m_center * m_center, axis=-1), axis=1)

    return {
        "phi_mean": phi_mean,
        "phi_std": np.std(phi, axis=1, ddof=1),
        "chi_phi": chi_phi,
        "chi_conn": chi_conn,
        "main_cost": np.mean(sdot, axis=1),
        "c_mis": np.mean(cmis, axis=1),
    }


def build_seed_metrics_dataframe(
    sim_out: Dict[str, np.ndarray],
    N: int,
    k: int,
    D: float,
    seed_ids: List[int],
    cfg: Exp3Config,
) -> Tuple[pd.DataFrame, Dict[int, Dict[str, Any]]]:
    basic = compute_basic_seed_metrics(sim_out, N)

    x_samples = sim_out["x_samples"]
    u_samples = sim_out["u_samples"]
    L = float(sim_out["L"])

    rows = []
    corr_payload: Dict[int, Dict[str, Any]] = {}
    for bi, seed in enumerate(seed_ids):
        corr = compute_seed_correlation_metrics(
            x_samples[bi],
            u_samples[bi],
            L=L,
            cfg=cfg,
            rng_seed=stable_boot_seed(N, k, D, seed, "corr"),
        )
        corr_payload[int(seed)] = corr

        chi_phi = float(basic["chi_phi"][bi])
        chi_conn = float(basic["chi_conn"][bi])
        main_cost = float(basic["main_cost"][bi])
        c_mis = float(basic["c_mis"][bi])
        chi_xi = float(corr["chi_xi"])
        chi_C = float(corr["chi_C"])
        xi = float(corr["xi"])
        c1 = main_cost / float(N)

        row = {
            "N": int(N),
            "k": int(k),
            "D": float(D),
            "seed": int(seed),
            "phi_mean": float(basic["phi_mean"][bi]),
            "phi_std": float(basic["phi_std"][bi]),
            "main_cost": main_cost,
            "c1": c1,
            "c_mis": c_mis,
            "chi_phi": chi_phi,
            "chi_conn": chi_conn,
            "xi": xi,
            "chi_xi": chi_xi,
            "chi_C": chi_C,
            "J_phi": main_cost / max(chi_phi, 1e-12),
            "J_conn": main_cost / max(chi_conn, 1e-12),
            "J_xi": main_cost / max(chi_xi, 1e-12),
            "J_C": main_cost / max(chi_C, 1e-12),
            "J_mis_phi": c_mis / max(chi_phi, 1e-12),
            "J_mis_conn": c_mis / max(chi_conn, 1e-12),
            "J_mis_xi": c_mis / max(chi_xi, 1e-12),
            "J_mis_C": c_mis / max(chi_C, 1e-12),
            "J_c1_phi": c1 / max(chi_phi, 1e-12),
            "J_c1_conn": c1 / max(chi_conn, 1e-12),
            "J_c1_xi": c1 / max(chi_xi, 1e-12),
            "J_c1_C": c1 / max(chi_C, 1e-12),
        }
        for alpha in (0.8, 1.0, 1.2):
            row[f"Jalpha_phi_{alpha:.1f}"] = main_cost / max(chi_phi ** alpha, 1e-12)
            row[f"Jalpha_conn_{alpha:.1f}"] = main_cost / max(chi_conn ** alpha, 1e-12)
            row[f"Jalpha_xi_{alpha:.1f}"] = main_cost / max(chi_xi ** alpha, 1e-12)
            row[f"Jalpha_C_{alpha:.1f}"] = main_cost / max(chi_C ** alpha, 1e-12)
        rows.append(row)

    return pd.DataFrame(rows), corr_payload




In [10]:
def run_one_condition(N: int, k: int, D: float, seeds: List[int], cfg: Exp3Config) -> Tuple[pd.DataFrame, Dict[int, Dict[str, Any]]]:
    batch_size = seed_batch_size_for_N(N, cfg)
    runner_cache: Dict[int, Any] = {}
    all_df = []
    all_corr: Dict[int, Dict[str, Any]] = {}

    for start in range(0, len(seeds), batch_size):
        seed_chunk = seeds[start:start + batch_size]
        B = len(seed_chunk)
        if B not in runner_cache:
            runner_cache[B] = make_condition_runner(N, k, B, cfg)
        runner = runner_cache[B]
        base_key = jr.PRNGKey(stable_boot_seed(N, k, D, tuple(seed_chunk), "run"))
        t0 = time.time()
        sim_out = runner(base_key, float(D))
        sim_out = jax.tree_util.tree_map(lambda x: np.asarray(jax.device_get(x)), sim_out)
        df_batch, corr_batch = build_seed_metrics_dataframe(sim_out, N, k, D, seed_chunk, cfg)
        all_df.append(df_batch)
        all_corr.update(corr_batch)
        print(f"  N={N} k={k} D={D:.6f} seeds {seed_chunk[0]}..{seed_chunk[-1]} done in {time.time()-t0:.2f}s")

    return pd.concat(all_df, ignore_index=True), all_corr




In [11]:
def aggregate_condition(df_cond: pd.DataFrame) -> pd.DataFrame:
    numeric_cols = [c for c in df_cond.columns if c not in ("N", "k", "D", "seed")]
    g = df_cond.groupby(["N", "k", "D"], as_index=False)
    mean_df = g[numeric_cols].mean()
    std_df = g[numeric_cols].std().add_suffix("_std")
    return mean_df.merge(std_df, on=["N", "k", "D"], how="left")


def local_quadratic_minimum(x: np.ndarray, y: np.ndarray) -> float:
    i = int(np.nanargmin(y))
    if i == 0 or i == len(x) - 1:
        return float(x[i])
    xs = np.asarray(x[i - 1:i + 2], dtype=float)
    ys = np.asarray(y[i - 1:i + 2], dtype=float)
    coeff = np.polyfit(xs, ys, deg=2)
    a, b, _ = coeff
    if abs(a) < 1e-12:
        return float(x[i])
    xstar = -b / (2.0 * a)
    if xstar < xs.min() or xstar > xs.max():
        return float(x[i])
    return float(xstar)


def pick_side_indices(i_opt: int, n: int, side_points: int) -> Tuple[int, int]:
    left = max(0, i_opt - side_points)
    right = min(n - 1, i_opt + side_points)
    if left == i_opt and i_opt + 1 < n:
        right = i_opt + 1
    if right == i_opt and i_opt - 1 >= 0:
        left = i_opt - 1
    return left, right


def bootstrap_curve_statistics(
    df_cond: pd.DataFrame,
    D_grid: List[float],
    chi_col: str,
    J_col: str,
    reps: int,
    valley_side_points: int,
    rng_seed: int,
) -> pd.DataFrame:
    rng = np.random.default_rng(rng_seed)
    perD = {float(D): df_cond[df_cond["D"] == D].copy() for D in D_grid}
    for D, sub in perD.items():
        if len(sub) == 0:
            raise ValueError(f"No rows found for D={D}")

    D_arr = np.array(D_grid, dtype=float)
    Dc_list = []
    Dopt_raw_list = []
    Dopt_smooth_list = []
    Delta_list = []
    Valley_list = []

    for _ in range(reps):
        chi_boot = []
        J_boot = []
        for D in D_grid:
            sub = perD[float(D)]
            idx = rng.integers(0, len(sub), size=len(sub))
            bs = sub.iloc[idx]
            chi_boot.append(float(bs[chi_col].mean()))
            J_boot.append(float(bs[J_col].mean()))
        chi_boot = np.asarray(chi_boot, dtype=float)
        J_boot = np.asarray(J_boot, dtype=float)

        i_c = int(np.nanargmax(chi_boot))
        i_opt = int(np.nanargmin(J_boot))
        D_c = float(D_arr[i_c])
        D_opt_raw = float(D_arr[i_opt])
        D_opt_smooth = float(local_quadratic_minimum(D_arr, J_boot))
        Delta = float(abs(D_c - D_opt_smooth))
        iL, iR = pick_side_indices(i_opt, len(D_arr), valley_side_points)
        valley = float((min(J_boot[iL], J_boot[iR]) - J_boot[i_opt]) / max(J_boot[i_opt], 1e-12))

        Dc_list.append(D_c)
        Dopt_raw_list.append(D_opt_raw)
        Dopt_smooth_list.append(D_opt_smooth)
        Delta_list.append(Delta)
        Valley_list.append(valley)

    def summarize(name: str, arr: List[float]) -> Dict[str, float]:
        a = np.asarray(arr, dtype=float)
        return {
            f"{name}_mean": float(np.mean(a)),
            f"{name}_q025": float(np.quantile(a, 0.025)),
            f"{name}_q975": float(np.quantile(a, 0.975)),
        }

    out: Dict[str, float] = {}
    out.update(summarize("Dc", Dc_list))
    out.update(summarize("Dopt_raw", Dopt_raw_list))
    out.update(summarize("Dopt_smooth", Dopt_smooth_list))
    out.update(summarize("Delta", Delta_list))
    out.update(summarize("Valley", Valley_list))
    return pd.DataFrame([out])




In [12]:
def propose_local_dense_grid(agg_df: pd.DataFrame, N: int, k: int, J_col: str, fine_points_per_side: int = 3) -> List[float]:
    sub = agg_df[(agg_df["N"] == N) & (agg_df["k"] == k)].sort_values("D")
    D = sub["D"].to_numpy(dtype=float)
    J = sub[J_col].to_numpy(dtype=float)
    i = int(np.nanargmin(J))
    step = np.median(np.diff(D)) if len(D) > 1 else 0.1
    fine_step = step / float(fine_points_per_side + 1)
    center = D[i]
    dense = [center + m * fine_step for m in range(-fine_points_per_side, fine_points_per_side + 1)]
    return sorted(set(round(x, 8) for x in list(D) + dense if x > 0))


def propose_boundary_extended_grid(D_grid: List[float], n_extend_each_side: int = 2, extension_step: Optional[float] = None) -> List[float]:
    D = np.array(sorted(D_grid), dtype=float)
    step = float(np.median(np.diff(D))) if extension_step is None and len(D) > 1 else float(extension_step or 0.1)
    left = [max(1e-6, D[0] - step * i) for i in range(n_extend_each_side, 0, -1)]
    right = [D[-1] + step * i for i in range(1, n_extend_each_side + 1)]
    return sorted(set(round(x, 8) for x in list(left) + list(D) + list(right)))




In [13]:
def run_scan(
    N_list: List[int],
    k_list: List[int],
    D_grid: List[float],
    seeds: List[int],
    cfg: Exp3Config,
    tag: str,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    dirs = ensure_dirs(cfg.output_dir)
    seed_csv = os.path.join(dirs["seed"], f"seed_metrics_{tag}.csv")
    agg_csv = os.path.join(dirs["agg"], f"aggregate_{tag}.csv")
    corr_json = os.path.join(dirs["meta"], f"corr_payload_{tag}.json")

    if cfg.resume and os.path.exists(seed_csv) and os.path.exists(agg_csv):
        df_seed = pd.read_csv(seed_csv)
        agg_df = pd.read_csv(agg_csv)
        corr_payload = {}
        if os.path.exists(corr_json):
            with open(corr_json, "r", encoding="utf-8") as f:
                corr_payload = json.load(f)
        return df_seed, agg_df, corr_payload

    all_seed_rows = []
    corr_payload: Dict[str, Any] = {}
    for N in N_list:
        for k in k_list:
            print(f"\n===== scan={tag} N={N} k={k} =====")
            for D in D_grid:
                df_cond, corr_cond = run_one_condition(N, k, float(D), seeds, cfg)
                all_seed_rows.append(df_cond)
                for seed_id, payload in corr_cond.items():
                    corr_payload[f"N{N}_k{k}_D{float(D):.8f}_seed{seed_id}"] = {
                        "r": np.asarray(payload["r"]).tolist(),
                        "C_r": np.asarray(payload["C_r"]).tolist(),
                        "xi": float(payload["xi"]),
                        "chi_xi": float(payload["chi_xi"]),
                        "chi_C": float(payload["chi_C"]),
                    }

    df_seed = pd.concat(all_seed_rows, ignore_index=True)
    agg_df = aggregate_condition(df_seed)
    df_seed.to_csv(seed_csv, index=False)
    agg_df.to_csv(agg_csv, index=False)
    with open(corr_json, "w", encoding="utf-8") as f:
        json.dump(corr_payload, f, ensure_ascii=False)
    return df_seed, agg_df, corr_payload




In [14]:
def plot_response_robustness(agg_df: pd.DataFrame, N: int, k: int, savepath: Optional[str] = None):
    sub = agg_df[(agg_df["N"] == N) & (agg_df["k"] == k)].sort_values("D")
    plt.figure(figsize=(7.0, 4.6))
    for col, label in [
        ("J_phi", r"$J_\phi = \dot S / \chi_\phi$"),
        ("J_conn", r"$J_{conn} = \dot S / \chi_{conn}$"),
        ("J_C", r"$J_C = \dot S / \chi_C$"),
        ("J_xi", r"$J_\xi = \dot S / \chi_\xi$"),
    ]:
        plt.plot(sub["D"], sub[col], marker="o", label=label)
    plt.xlabel("D")
    plt.ylabel("cost / response")
    plt.title(f"Experiment 3A: response-definition robustness (N={N}, k={k})")
    plt.legend()
    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=160, bbox_inches="tight")
        plt.close()
    else:
        plt.show()


def plot_alpha_robustness(agg_df: pd.DataFrame, N: int, k: int, response_key: str = "phi", savepath: Optional[str] = None):
    sub = agg_df[(agg_df["N"] == N) & (agg_df["k"] == k)].sort_values("D")
    plt.figure(figsize=(7.0, 4.6))
    for alpha in (0.8, 1.0, 1.2):
        col = f"Jalpha_{response_key}_{alpha:.1f}"
        plt.plot(sub["D"], sub[col], marker="o", label=rf"$\alpha={alpha:.1f}$")
    plt.xlabel("D")
    plt.ylabel(r"$J_\alpha = \dot S / \chi^\alpha$")
    plt.title(f"Experiment 3B: exponent robustness ({response_key}, N={N}, k={k})")
    plt.legend()
    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=160, bbox_inches="tight")
        plt.close()
    else:
        plt.show()


def plot_cost_robustness(agg_df: pd.DataFrame, N: int, k: int, response_key: str = "phi", savepath: Optional[str] = None):
    sub = agg_df[(agg_df["N"] == N) & (agg_df["k"] == k)].sort_values("D")
    plt.figure(figsize=(7.0, 4.6))
    mapping = [
        (f"J_{response_key}", r"main $\dot S$"),
        (f"J_mis_{response_key}", r"$c_{mis}$"),
        (f"J_c1_{response_key}", r"$c_1 = \dot S / N$"),
    ]
    for col, label in mapping:
        if col in sub.columns:
            plt.plot(sub["D"], sub[col], marker="o", label=label)
    plt.xlabel("D")
    plt.ylabel("cost / response")
    plt.title(f"Experiment 3B: cost-definition robustness ({response_key}, N={N}, k={k})")
    plt.legend()
    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=160, bbox_inches="tight")
        plt.close()
    else:
        plt.show()


def plot_bootstrap_summary(boot_df: pd.DataFrame, title: str, savepath: Optional[str] = None):
    row = boot_df.iloc[0]
    names = ["Dc", "Dopt_smooth", "Delta", "Valley"]
    means = [row[f"{n}_mean"] for n in names]
    lows = [row[f"{n}_mean"] - row[f"{n}_q025"] for n in names]
    highs = [row[f"{n}_q975"] - row[f"{n}_mean"] for n in names]
    x = np.arange(len(names))
    plt.figure(figsize=(7.0, 4.3))
    plt.errorbar(x, means, yerr=[lows, highs], fmt="o")
    plt.xticks(x, names)
    plt.title(title)
    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=160, bbox_inches="tight")
        plt.close()
    else:
        plt.show()


def plot_local_valley(agg_df: pd.DataFrame, N: int, k: int, J_col: str = "J_phi", savepath: Optional[str] = None):
    sub = agg_df[(agg_df["N"] == N) & (agg_df["k"] == k)].sort_values("D")
    D = sub["D"].to_numpy(dtype=float)
    J = sub[J_col].to_numpy(dtype=float)
    D_s = local_quadratic_minimum(D, J)
    plt.figure(figsize=(7.0, 4.3))
    plt.plot(D, J, marker="o", label=J_col)
    plt.axvline(D_s, linestyle="--", label=f"smooth opt = {D_s:.4f}")
    plt.xlabel("D")
    plt.ylabel(J_col)
    plt.title(f"Experiment 3D: local valley check (N={N}, k={k})")
    plt.legend()
    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=160, bbox_inches="tight")
        plt.close()
    else:
        plt.show()




In [15]:
def run_full_experiment3_pipeline(
    N_list: List[int],
    k_list: List[int],
    D_coarse: List[float],
    seeds: List[int],
    cfg: Exp3Config,
    bootstrap_response_key: str = "phi",
):
    report_jax_backend(require_gpu=cfg.require_gpu)
    dirs = ensure_dirs(cfg.output_dir)

    manifest = {
        "config": asdict(cfg),
        "N_list": list(N_list),
        "k_list": list(k_list),
        "D_coarse": list(D_coarse),
        "seeds": list(seeds),
        "jax_devices": report_jax_backend(require_gpu=False),
    }
    with open(os.path.join(dirs["meta"], "manifest_exp3.json"), "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    # 1) coarse scan
    seed_coarse, agg_coarse, _ = run_scan(N_list, k_list, D_coarse, seeds, cfg, tag="coarse")

    # 2) local dense scan per (N, k)
    dense_union: List[float] = []
    for N in N_list:
        for k in k_list:
            dense_union.extend(propose_local_dense_grid(agg_coarse, N, k, J_col=f"J_{bootstrap_response_key}", fine_points_per_side=3))
    D_dense = sorted(set(round(x, 8) for x in dense_union))
    print("Dense grid:", D_dense)
    seed_dense, agg_dense, _ = run_scan(N_list, k_list, D_dense, seeds, cfg, tag="dense")

    # 3) boundary extension on union grid
    D_extended = propose_boundary_extended_grid(D_dense, n_extend_each_side=2)
    print("Boundary-extended grid:", D_extended)
    seed_ext, agg_ext, _ = run_scan(N_list, k_list, D_extended, seeds, cfg, tag="extended")

    # 4) bootstrap summary
    boot_rows = []
    for N in N_list:
        for k in k_list:
            sub = seed_ext[(seed_ext["N"] == N) & (seed_ext["k"] == k)].copy()
            grid = sorted(sub["D"].unique().tolist())
            boot = bootstrap_curve_statistics(
                sub,
                D_grid=grid,
                chi_col=f"chi_{bootstrap_response_key}",
                J_col=f"J_{bootstrap_response_key}",
                reps=cfg.bootstrap_reps,
                valley_side_points=cfg.valley_side_points,
                rng_seed=stable_boot_seed(N, k, bootstrap_response_key, "boot"),
            )
            boot.insert(0, "N", N)
            boot.insert(1, "k", k)
            boot.insert(2, "response_key", bootstrap_response_key)
            boot_rows.append(boot)

            plot_response_robustness(agg_ext, N, k, savepath=os.path.join(dirs["fig"], f"fig3a_response_N{N}_k{k}.png"))
            plot_cost_robustness(agg_ext, N, k, response_key=bootstrap_response_key, savepath=os.path.join(dirs["fig"], f"fig3b_cost_N{N}_k{k}.png"))
            plot_alpha_robustness(agg_ext, N, k, response_key=bootstrap_response_key, savepath=os.path.join(dirs["fig"], f"fig3b_alpha_N{N}_k{k}.png"))
            plot_local_valley(agg_ext, N, k, J_col=f"J_{bootstrap_response_key}", savepath=os.path.join(dirs["fig"], f"fig3d_valley_N{N}_k{k}.png"))

    boot_df = pd.concat(boot_rows, ignore_index=True)
    boot_csv = os.path.join(dirs["boot"], "bootstrap_summary.csv")
    boot_df.to_csv(boot_csv, index=False)

    if len(boot_df) > 0:
        plot_bootstrap_summary(
            boot_df.iloc[[0]],
            title="Experiment 3 bootstrap summary",
            savepath=os.path.join(dirs["fig"], "fig3c_bootstrap.png"),
        )

    print("\nAll outputs saved to:", os.path.abspath(cfg.output_dir))
    return {
        "seed_coarse": seed_coarse,
        "agg_coarse": agg_coarse,
        "seed_dense": seed_dense,
        "agg_dense": agg_dense,
        "seed_extended": seed_ext,
        "agg_extended": agg_ext,
        "bootstrap": boot_df,
    }




In [16]:
def benchmark_single_condition():
    cfg = Exp3Config(
        burnin_steps=300,
        sample_stride=10,
        n_samples=20,
        bootstrap_reps=100,
        output_dir="results_exp3_gpu_jax_smoke",
        require_gpu=False,
    )
    report_jax_backend(require_gpu=False)
    df, _ = run_one_condition(N=128, k=6, D=0.08, seeds=[0, 1], cfg=cfg)
    print(df.head())
    return df




In [17]:
# Notebook quick-start examples:
#
# cfg = Exp3Config(
#     burnin_steps=5000,
#     sample_stride=50,
#     n_samples=160,
#     bootstrap_reps=2000,
#     output_dir="results_exp3_gpu_jax",
# )
#
# results = run_full_experiment3_pipeline(
#     N_list=[1024],
#     k_list=[5, 7, 9, 12],
#     D_coarse=[0.005, 0.02, 0.05, 0.0738, 0.10, 0.15],
#     seeds=list(range(8)),
#     cfg=cfg,
#     bootstrap_response_key="phi",
# )
#
# For 2048 on a 5090, reduce seed batch size if you hit memory pressure.




In [18]:
def _running_in_ipython() -> bool:
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except Exception:
        return False


def main_cli():
    import argparse

    parser = argparse.ArgumentParser(description="Experiment 3 aligned with Exp1+Exp2 definitions")
    parser.add_argument("--quick", action="store_true", help="Run smoke benchmark")
    parser.add_argument("--formal", action="store_true", help="Run a formal starter configuration")
    parser.add_argument("--output-dir", default="results_exp3_gpu_jax")
    args, unknown = parser.parse_known_args()

    if unknown:
        print("Ignored extra args:", unknown)

    if args.quick:
        benchmark_single_condition()
        return

    if args.formal:
        cfg = Exp3Config(output_dir=args.output_dir, require_gpu=True)
        run_full_experiment3_pipeline(
            N_list=[1024],
            k_list=[5, 7, 9, 12],
            D_coarse=[0.005, 0.02, 0.05, 0.0738, 0.10, 0.15],
            seeds=list(range(8)),
            cfg=cfg,
            bootstrap_response_key="phi",
        )
        return

    parser.print_help()


RUN_MODE = None  # set to "quick" or "formal" in notebook if desired
OUTPUT_DIR = "results_exp3_gpu_jax"

if _running_in_ipython():
    if RUN_MODE == "quick":
        benchmark_single_condition()
    elif RUN_MODE == "formal":
        cfg = Exp3Config(output_dir=OUTPUT_DIR, require_gpu=True)
        run_full_experiment3_pipeline(
            N_list=[1024],
            k_list=[5, 7, 9, 12],
            D_coarse=[0.005, 0.02, 0.05, 0.0738, 0.10, 0.15],
            seeds=list(range(8)),
            cfg=cfg,
            bootstrap_response_key="phi",
        )
elif __name__ == "__main__":
    main_cli()


In [20]:
def aggregate_condition(df_cond: pd.DataFrame) -> pd.DataFrame:
    numeric_cols = [c for c in df_cond.columns if c not in ("N", "k", "D", "seed")]
    keys = ["N", "k", "D"]

    mean_df = (
        df_cond.groupby(keys, as_index=False)[numeric_cols]
        .mean()
    )

    std_df = (
        df_cond.groupby(keys, as_index=False)[numeric_cols]
        .std()
        .rename(columns={c: f"{c}_std" for c in numeric_cols})
    )

    return mean_df.merge(std_df, on=keys, how="left")

In [22]:
def _positive_response_floor(x, eps: float = 1e-12) -> float:
    x = np.real_if_close(x)
    try:
        x = float(x)
    except Exception:
        return eps
    if (not np.isfinite(x)) or (x <= eps):
        return eps
    return x


def build_seed_metrics_dataframe(
    sim_out: Dict[str, np.ndarray],
    N: int,
    k: int,
    D: float,
    seed_ids: List[int],
    cfg: Exp3Config,
) -> Tuple[pd.DataFrame, Dict[int, Dict[str, Any]]]:
    basic = compute_basic_seed_metrics(sim_out, N)

    x_samples = sim_out["x_samples"]
    u_samples = sim_out["u_samples"]
    L = float(sim_out["L"])

    rows = []
    corr_payload: Dict[int, Dict[str, Any]] = {}

    for bi, seed in enumerate(seed_ids):
        corr = compute_seed_correlation_metrics(
            x_samples[bi],
            u_samples[bi],
            L=L,
            cfg=cfg,
            rng_seed=stable_boot_seed(N, k, D, seed, "corr"),
        )
        corr_payload[int(seed)] = corr

        # raw values: 原样保留，方便你后面检查是否真的出现了微小负值
        chi_phi_raw = float(basic["chi_phi"][bi])
        chi_conn_raw = float(basic["chi_conn"][bi])
        main_cost = float(basic["main_cost"][bi])
        c_mis = float(basic["c_mis"][bi])
        chi_xi_raw = float(corr["chi_xi"])
        chi_C_raw = float(corr["chi_C"])
        xi = float(corr["xi"])
        c1 = main_cost / float(N)

        # safe denominators: 只在构造 J 时做正值下限截断
        chi_phi = _positive_response_floor(chi_phi_raw)
        chi_conn = _positive_response_floor(chi_conn_raw)
        chi_xi = _positive_response_floor(chi_xi_raw)
        chi_C = _positive_response_floor(chi_C_raw)

        row = {
            "N": int(N),
            "k": int(k),
            "D": float(D),
            "seed": int(seed),

            "phi_mean": float(basic["phi_mean"][bi]),
            "phi_std": float(basic["phi_std"][bi]),

            "main_cost": main_cost,
            "c1": c1,
            "c_mis": c_mis,

            # raw observables 保留
            "chi_phi": chi_phi_raw,
            "chi_conn": chi_conn_raw,
            "xi": xi,
            "chi_xi": chi_xi_raw,
            "chi_C": chi_C_raw,

            # main J
            "J_phi": main_cost / chi_phi,
            "J_conn": main_cost / chi_conn,
            "J_xi": main_cost / chi_xi,
            "J_C": main_cost / chi_C,

            # alternative costs
            "J_mis_phi": c_mis / chi_phi,
            "J_mis_conn": c_mis / chi_conn,
            "J_mis_xi": c_mis / chi_xi,
            "J_mis_C": c_mis / chi_C,

            "J_c1_phi": c1 / chi_phi,
            "J_c1_conn": c1 / chi_conn,
            "J_c1_xi": c1 / chi_xi,
            "J_c1_C": c1 / chi_C,
        }

        for alpha in (0.8, 1.0, 1.2):
            row[f"Jalpha_phi_{alpha:.1f}"] = main_cost / (chi_phi ** alpha)
            row[f"Jalpha_conn_{alpha:.1f}"] = main_cost / (chi_conn ** alpha)
            row[f"Jalpha_xi_{alpha:.1f}"] = main_cost / (chi_xi ** alpha)
            row[f"Jalpha_C_{alpha:.1f}"] = main_cost / (chi_C ** alpha)

        rows.append(row)

    return pd.DataFrame(rows), corr_payload

In [23]:
cfg = Exp3Config(
    burnin_steps=5000,
    sample_stride=50,
    n_samples=160,
    bootstrap_reps=2000,
    output_dir="results_exp3_gpu_jax",
)

results = run_full_experiment3_pipeline(
    N_list=[1024],
    k_list=[5, 7, 9, 12],
    D_coarse=[0.005, 0.02, 0.05, 0.0738, 0.10, 0.15],
    seeds=list(range(8)),
    cfg=cfg,
    bootstrap_response_key="phi",
)

JAX devices: ['gpu:NVIDIA GeForce RTX 5090']
JAX devices: ['gpu:NVIDIA GeForce RTX 5090']
Dense grid: [np.float64(0.005), np.float64(0.02), np.float64(0.05), np.float64(0.05415), np.float64(0.0607), np.float64(0.06725), np.float64(0.0738), np.float64(0.08035), np.float64(0.0869), np.float64(0.09345), np.float64(0.1), np.float64(0.13035), np.float64(0.1369), np.float64(0.14345), np.float64(0.15), np.float64(0.15655), np.float64(0.1631), np.float64(0.16965)]
Boundary-extended grid: [1e-06, np.float64(0.005), np.float64(0.02), np.float64(0.05), np.float64(0.05415), np.float64(0.0607), np.float64(0.06725), np.float64(0.0738), np.float64(0.08035), np.float64(0.0869), np.float64(0.09345), np.float64(0.1), np.float64(0.13035), np.float64(0.1369), np.float64(0.14345), np.float64(0.15), np.float64(0.15655), np.float64(0.1631), np.float64(0.16965), np.float64(0.1762), np.float64(0.18275)]

===== scan=extended N=1024 k=5 =====
  N=1024 k=5 D=0.000001 seeds 0..7 done in 2.79s
  N=1024 k=5 D=0.0050

In [24]:
import os
import numpy as np
import pandas as pd


def _resp_j_cols(response_key: str):
    mapping = {
        "phi":  ("chi_phi",  "J_phi"),
        "conn": ("chi_conn", "J_conn"),
        "xi":   ("chi_xi",   "J_xi"),
        "C":    ("chi_C",    "J_C"),
    }
    if response_key not in mapping:
        raise ValueError(f"Unknown response_key={response_key}")
    return mapping[response_key]


def _valley_depth(j_curve: np.ndarray, i_opt: int, side_points: int = 2):
    n = len(j_curve)
    left_i = max(0, i_opt - side_points)
    right_i = min(n - 1, i_opt + side_points)

    if left_i == i_opt or right_i == i_opt:
        return np.nan

    j_opt = float(j_curve[i_opt])
    j_left = float(j_curve[left_i])
    j_right = float(j_curve[right_i])

    if (not np.isfinite(j_opt)) or j_opt <= 0:
        return np.nan
    if (not np.isfinite(j_left)) or (not np.isfinite(j_right)):
        return np.nan

    shoulder = min(j_left, j_right)
    return (shoulder - j_opt) / j_opt


def rerun_bootstrap_from_saved_seed_metrics(
    output_dir="results_exp3_gpu_jax",
    source_tag="extended",
    response_key="conn",
    bootstrap_reps=2000,
    valley_side_points=2,
    rng_seed=2026,
    save_name=None,
):
    seed_csv = os.path.join(output_dir, "seed_metrics", f"seed_metrics_{source_tag}.csv")
    if not os.path.exists(seed_csv):
        raise FileNotFoundError(f"Not found: {seed_csv}")

    df = pd.read_csv(seed_csv)
    resp_col, j_col = _resp_j_cols(response_key)

    need_cols = ["N", "k", "D", "seed", resp_col, j_col]
    missing = [c for c in need_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in seed metrics: {missing}")

    rng = np.random.default_rng(rng_seed)
    rows = []

    for (N, k), g in df.groupby(["N", "k"]):
        g = g.copy()

        resp_pivot = g.pivot(index="D", columns="seed", values=resp_col).sort_index()
        j_pivot = g.pivot(index="D", columns="seed", values=j_col).sort_index()

        D_vals = resp_pivot.index.to_numpy(dtype=float)
        resp_mat = resp_pivot.to_numpy(dtype=float)
        j_mat = j_pivot.to_numpy(dtype=float)

        if resp_mat.shape != j_mat.shape:
            raise ValueError(f"Shape mismatch for N={N}, k={k}")

        n_D, n_seeds = resp_mat.shape

        # raw
        resp_mean_raw = np.nanmean(resp_mat, axis=1)
        j_mean_raw = np.nanmean(j_mat, axis=1)

        i_dc_raw = int(np.nanargmax(resp_mean_raw))
        i_dopt_raw = int(np.nanargmin(j_mean_raw))

        D_c_raw = float(D_vals[i_dc_raw])
        D_opt_raw = float(D_vals[i_dopt_raw])
        Delta_raw = abs(D_c_raw - D_opt_raw)
        Valley_raw = _valley_depth(j_mean_raw, i_dopt_raw, side_points=valley_side_points)

        # bootstrap
        boot_Dc = np.empty(bootstrap_reps, dtype=float)
        boot_Dopt = np.empty(bootstrap_reps, dtype=float)
        boot_Delta = np.empty(bootstrap_reps, dtype=float)
        boot_Valley = np.empty(bootstrap_reps, dtype=float)

        for b in range(bootstrap_reps):
            idx = rng.integers(0, n_seeds, size=n_seeds)

            resp_b = np.nanmean(resp_mat[:, idx], axis=1)
            j_b = np.nanmean(j_mat[:, idx], axis=1)

            i_dc = int(np.nanargmax(resp_b))
            i_dopt = int(np.nanargmin(j_b))

            D_c = float(D_vals[i_dc])
            D_opt = float(D_vals[i_dopt])

            boot_Dc[b] = D_c
            boot_Dopt[b] = D_opt
            boot_Delta[b] = abs(D_c - D_opt)
            boot_Valley[b] = _valley_depth(j_b, i_dopt, side_points=valley_side_points)

        def _mean_ci(x):
            x = np.asarray(x, dtype=float)
            x = x[np.isfinite(x)]
            if len(x) == 0:
                return np.nan, np.nan, np.nan
            return float(np.mean(x)), float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5))

        Dc_mean, Dc_lo, Dc_hi = _mean_ci(boot_Dc)
        Dopt_mean, Dopt_lo, Dopt_hi = _mean_ci(boot_Dopt)
        Delta_mean, Delta_lo, Delta_hi = _mean_ci(boot_Delta)
        Valley_mean, Valley_lo, Valley_hi = _mean_ci(boot_Valley)

        rows.append({
            "response_key": response_key,
            "source_tag": source_tag,
            "N": int(N),
            "k": int(k),

            "D_c_raw": D_c_raw,
            "D_opt_raw": D_opt_raw,
            "Delta_raw": Delta_raw,
            "Valley_raw": Valley_raw,

            "D_c_mean": Dc_mean,
            "D_c_ci_lo": Dc_lo,
            "D_c_ci_hi": Dc_hi,

            "D_opt_mean": Dopt_mean,
            "D_opt_ci_lo": Dopt_lo,
            "D_opt_ci_hi": Dopt_hi,

            "Delta_mean": Delta_mean,
            "Delta_ci_lo": Delta_lo,
            "Delta_ci_hi": Delta_hi,

            "Valley_mean": Valley_mean,
            "Valley_ci_lo": Valley_lo,
            "Valley_ci_hi": Valley_hi,

            "n_D": int(n_D),
            "n_seeds": int(n_seeds),
            "bootstrap_reps": int(bootstrap_reps),
        })

    out = pd.DataFrame(rows).sort_values(["N", "k"]).reset_index(drop=True)

    if save_name is None:
        save_name = f"bootstrap_summary_{response_key}_from_{source_tag}.csv"

    save_path = os.path.join(output_dir, save_name)
    out.to_csv(save_path, index=False)

    print("saved:", save_path)
    display(out)
    return out

In [26]:
bootstrap_conn = rerun_bootstrap_from_saved_seed_metrics(
    output_dir="results_exp3_gpu_jax",
    source_tag="extended",      # 直接用 extended，因为它已经是 union grid
    response_key="conn",        # 关键：改成 conn
    bootstrap_reps=2000,
    valley_side_points=2,
    rng_seed=2026,
)

saved: results_exp3_gpu_jax/bootstrap_summary_conn_from_extended.csv


,response_key,source_tag,N,k,D_c_raw,D_opt_raw,Delta_raw,Valley_raw,D_c_mean,D_c_ci_lo,...,D_opt_ci_hi,Delta_mean,Delta_ci_lo,Delta_ci_hi,Valley_mean,Valley_ci_lo,Valley_ci_hi,n_D,n_seeds,bootstrap_reps
0,conn,extended,1024,5,0.05000,0.0500,0.00000,0.303174,0.035570,0.02,...,0.050000,0.007560,0.0,0.03000,0.284889,0.120510,0.373858,21,8,2000
1,conn,extended,1024,7,0.05415,0.0738,0.01965,0.104781,0.061646,0.05,...,0.080350,0.004565,0.0,0.02620,0.158379,0.008169,0.394002,21,8,2000
2,conn,extended,1024,9,0.07380,0.0869,0.01310,0.218892,0.076645,0.05,...,0.087064,0.006768,0.0,0.03690,0.216449,0.009962,0.561751,21,8,2000
3,conn,extended,1024,12,0.05000,0.1000,0.05000,0.061371,0.068059,0.05,...,0.176200,0.038936,0.0,0.12205,0.219418,0.013410,0.789023,21,8,2000


In [27]:
D_extra_high = [0.20, 0.22, 0.25, 0.28]

seed_extra, agg_extra, corr_extra = run_scan(
    N_list=[1024],
    k_list=[7, 9, 12],
    D_grid=D_extra_high,
    seeds=list(range(8)),
    cfg=cfg,
    tag="highD_extra",
)


===== scan=highD_extra N=1024 k=7 =====
  N=1024 k=7 D=0.200000 seeds 0..7 done in 2.75s
  N=1024 k=7 D=0.220000 seeds 0..7 done in 2.71s
  N=1024 k=7 D=0.250000 seeds 0..7 done in 2.64s
  N=1024 k=7 D=0.280000 seeds 0..7 done in 2.65s

===== scan=highD_extra N=1024 k=9 =====
  N=1024 k=9 D=0.200000 seeds 0..7 done in 4.18s
  N=1024 k=9 D=0.220000 seeds 0..7 done in 4.14s
  N=1024 k=9 D=0.250000 seeds 0..7 done in 4.17s
  N=1024 k=9 D=0.280000 seeds 0..7 done in 4.20s

===== scan=highD_extra N=1024 k=12 =====
  N=1024 k=12 D=0.200000 seeds 0..7 done in 4.16s
  N=1024 k=12 D=0.220000 seeds 0..7 done in 4.16s
  N=1024 k=12 D=0.250000 seeds 0..7 done in 4.18s
  N=1024 k=12 D=0.280000 seeds 0..7 done in 4.20s


In [28]:
import os
import pandas as pd

base = "results_exp3_gpu_jax"

old_seed = pd.read_csv(os.path.join(base, "seed_metrics", "seed_metrics_extended.csv"))
new_seed = pd.read_csv(os.path.join(base, "seed_metrics", "seed_metrics_highD_extra.csv"))

merged_seed = (
    pd.concat([old_seed, new_seed], axis=0)
    .drop_duplicates(subset=["N", "k", "D", "seed"], keep="last")
    .sort_values(["N", "k", "D", "seed"])
    .reset_index(drop=True)
)

merged_path = os.path.join(base, "seed_metrics", "seed_metrics_extended_plus_highD.csv")
merged_seed.to_csv(merged_path, index=False)
print("saved:", merged_path)

saved: results_exp3_gpu_jax/seed_metrics/seed_metrics_extended_plus_highD.csv


In [29]:
bootstrap_phi_plus = rerun_bootstrap_from_saved_seed_metrics(
    output_dir="results_exp3_gpu_jax",
    source_tag="extended_plus_highD",   # 用刚合并的新文件
    response_key="phi",
    bootstrap_reps=2000,
    valley_side_points=2,
    rng_seed=2026,
    save_name="bootstrap_summary_phi_from_extended_plus_highD.csv",
)

saved: results_exp3_gpu_jax/bootstrap_summary_phi_from_extended_plus_highD.csv


,response_key,source_tag,N,k,D_c_raw,D_opt_raw,Delta_raw,Valley_raw,D_c_mean,D_c_ci_lo,...,D_opt_ci_hi,Delta_mean,Delta_ci_lo,Delta_ci_hi,Valley_mean,Valley_ci_lo,Valley_ci_hi,n_D,n_seeds,bootstrap_reps
0,phi,extended_plus_highD,1024,5,0.06725,0.06725,0.00000,0.052820,0.062579,0.05000,...,0.09345,0.006478,0.0,0.039300,0.078506,0.002939,0.249755,21,8,2000
1,phi,extended_plus_highD,1024,7,0.15655,0.20000,0.04345,0.080164,0.166217,0.15655,...,0.22000,0.017122,0.0,0.050514,0.071186,0.002336,0.194267,25,8,2000
2,phi,extended_plus_highD,1024,9,0.28000,0.28000,0.00000,NaN,0.280000,0.28000,...,0.28000,0.000000,0.0,0.000000,NaN,NaN,NaN,25,8,2000
3,phi,extended_plus_highD,1024,12,0.28000,0.28000,0.00000,NaN,0.276460,0.25000,...,0.28000,0.002385,0.0,0.030000,0.057620,0.002137,0.197067,25,8,2000
